# 서비스 모델 추론 확인

실제 학습된 YOLO11n 960 가중치와 작업자·보호구 연결 코드를 train 이미지 한 장으로 검사합니다. 모델 학습과 test 평가는 다시 하지 않습니다.


In [ ]:
!pip -q install ultralytics==8.4.208
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import zipfile
from PIL import Image
from ultralytics import YOLO
from IPython.display import display


In [ ]:
WEIGHTS = Path("/content/drive/MyDrive/ppe-project/runs/improved_yolo11n_img960_seed42/weights/best.pt")
ZIP = Path("/content/drive/MyDrive/construction-ppe-5class.zip")
assert WEIGHTS.is_file() and ZIP.is_file(), "Drive의 모델 또는 데이터 ZIP을 찾지 못했습니다."
model = YOLO(str(WEIGHTS))
expected = ["Person", "helmet", "no_helmet", "vest", "no_vest"]
assert [model.names[i] for i in range(5)] == expected, model.names
print("모델 클래스 확인:", model.names)
with zipfile.ZipFile(ZIP) as z:
    name = "construction-ppe-5class/images/train/ultralytics-ppe-nov2026-009.jpg"
    with z.open(name) as stream:
        image = Image.open(stream).convert("RGB")
result = model.predict(image, imgsz=960, conf=0.25, verbose=False)[0]
detections = [{"name": result.names[int(b.cls.item())], "confidence": float(b.conf.item()), "box": b.xyxy[0].tolist()} for b in result.boxes]
print("탐지 수:", len(detections))
print([(d["name"], round(d["confidence"], 2)) for d in detections])
display(Image.fromarray(result.plot()[:, :, ::-1]).resize((640,640)))


In [ ]:
%%writefile safety_logic.py
"""Geometry and conservative PPE status rules for the educational demo."""

def contains(point, polygon):
    x, y = point
    inside = False
    for i, (x1, y1) in enumerate(polygon):
        x2, y2 = polygon[(i + 1) % len(polygon)]
        cross = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)
        if abs(cross) < 1e-9 and min(x1, x2) <= x <= max(x1, x2) and min(y1, y2) <= y <= max(y1, y2):
            return True
        if (y1 > y) != (y2 > y) and x < (x2 - x1) * (y - y1) / (y2 - y1) + x1:
            inside = not inside
    return inside


def associate(detections, polygon):
    people = [dict(d, gear=[]) for d in detections if d['name'] == 'Person']
    unassigned = []
    for item in (d for d in detections if d['name'] != 'Person'):
        x1, y1, x2, y2 = item['box']
        center = ((x1+x2)/2, (y1+y2)/2)
        matches = [p for p in people if p['box'][0] <= center[0] <= p['box'][2] and p['box'][1] <= center[1] <= p['box'][3]]
        if matches:
            # Smallest containing person limits ambiguous overlaps, but is still heuristic.
            min(matches, key=lambda p: (p['box'][2]-p['box'][0])*(p['box'][3]-p['box'][1]))['gear'].append(item['name'])
        else:
            unassigned.append(item)
    for person in people:
        x1, y1, x2, y2 = person['box']
        person['foot'] = ((x1+x2)/2, y2)
        person['intrusion'] = bool(polygon) and contains(person['foot'], polygon)
        for key, positive, negative in [('helmet', 'helmet', 'no_helmet'), ('vest', 'vest', 'no_vest')]:
            present, absent = positive in person['gear'], negative in person['gear']
            person[key] = '충돌' if present and absent else '미착용' if absent else '착용' if present else '확인 불가'
    return people, unassigned


def overall(people, unassigned, zone_enabled):
    if any(p['intrusion'] for p in people):
        return 'DANGER'
    if any(p[k] in ('미착용', '충돌') for p in people for k in ('helmet', 'vest')) or any(d['name'].startswith('no_') for d in unassigned):
        return 'WARNING'
    if not people or not zone_enabled or unassigned or any(p[k] == '확인 불가' for p in people for k in ('helmet', 'vest')):
        return '확인 필요'
    return 'SAFE'



In [ ]:
from safety_logic import associate, overall
w,h=image.size
zone=[(w*.2,h*.4),(w*.8,h*.4),(w*.8,h*.9),(w*.2,h*.9)]
people, unassigned = associate(detections, zone)
print("작업자 수:", len(people), "전체 상태:", overall(people, unassigned, True))
for i,p in enumerate(people,1):
    print("작업자", i, "안전모:", p["helmet"], "안전조끼:", p["vest"], "위험구역 침입:", p["intrusion"])
print("연결되지 않은 보호구:",len(unassigned))
